In [ ]:
# ==============================================================================
# 1. INSTALLS & IMPORTS
# ==============================================================================
!pip install pymupdf ollama scikit-learn nltk huggingface_hub
!sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd rsync curl lshw

import pymupdf as fitz
import json, requests, datetime, numpy as np
from pathlib import Path
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
import os, re, subprocess, shutil, time, threading, nltk, random, hashlib
from ollama import Client

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==============================================================================
# 2. PATH MAPPING & RESTORATION
# ==============================================================================
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/research-assistant")

DRIVE_WORKSPACE = DRIVE_DIR / "workspace"
DRIVE_PERSONALITY = DRIVE_DIR / "personality"
DRIVE_MODELS = DRIVE_DIR / "models"

DRIVE_MEMORY = DRIVE_PERSONALITY / "memories"
DRIVE_LOGS = DRIVE_PERSONALITY / "logs"
DRIVE_QUERIES = DRIVE_PERSONALITY / "queries"
DRIVE_PENDING = DRIVE_QUERIES / "pending"    # cloud writes, forager consumes
DRIVE_CLAIMED = DRIVE_QUERIES / "claimed"    # forager: in progress
DRIVE_DONE = DRIVE_QUERIES / "done"          # forager: finished (this is the ledger)

DRIVE_UNPROCESSED = DRIVE_WORKSPACE / "unprocessed"
DRIVE_PROCESSED = DRIVE_WORKSPACE / "processed"
DRIVE_TEMP = DRIVE_WORKSPACE / "temp-chunks"

LOCAL_DIR = Path("/content")

LOCAL_OLLAMA = Path("/content/research-ollama-models")
LOCAL_MODELS = Path("/content/research-models")
LOCAL_WORKSPACE = Path("/content/research-workspace")
LOCAL_PERSONALITY = Path("/content/research-personality")

LOCAL_MEMORY = LOCAL_PERSONALITY / "memories"
LOCAL_LOGS = LOCAL_PERSONALITY / "logs"
LOCAL_QUERIES = LOCAL_PERSONALITY / "queries"

LOCAL_TEMP = LOCAL_WORKSPACE / "temp-chunks"
LOCAL_PROCESSED = LOCAL_WORKSPACE / "processed"
LOCAL_UNPROCESSED = LOCAL_WORKSPACE / "unprocessed"

directories = [
    LOCAL_TEMP, LOCAL_OLLAMA, LOCAL_WORKSPACE, LOCAL_PERSONALITY,
    LOCAL_MEMORY, LOCAL_LOGS, LOCAL_QUERIES, DRIVE_MEMORY, DRIVE_LOGS, DRIVE_QUERIES,
    DRIVE_PENDING, DRIVE_CLAIMED, DRIVE_DONE,
    LOCAL_PROCESSED, DRIVE_UNPROCESSED, DRIVE_PROCESSED, DRIVE_TEMP, DRIVE_MODELS,
    DRIVE_WORKSPACE, DRIVE_PERSONALITY
]
for p in directories: p.mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------------------
# 2b. File names
# ------------------------------------------------------------------------------
# Steering files (written by the setup notebook; the memory file is rewritten here as the research focus evolves)
MEMORY_FILE = "MEMORY.md"
INSTRUCTIONS_FILE = "INSTRUCTIONS.md"
NUDGE_FILE = "NUDGE.md"
DRIVE_MEMORY_FILE = DRIVE_PERSONALITY / MEMORY_FILE
DRIVE_NUDGE_FILE = DRIVE_PERSONALITY / NUDGE_FILE
LOCAL_MEMORY_FILE = LOCAL_PERSONALITY / MEMORY_FILE
LOCAL_INSTRUCTIONS_FILE = LOCAL_PERSONALITY / INSTRUCTIONS_FILE

# Memories
COMPILED_MEMORIES_FILE = LOCAL_MEMORY / "compiled_memories.md"
FOCUS_HISTORY_FILE = LOCAL_MEMORY / "evolved_research_focuses.md"
META_SYNTHESIS_TEMPLATE = "meta_synthesis_{timestamp}.md"
META_SYNTHESIS_GLOB = "meta_synthesis_*.md"

# Chunks and logs
CHUNK_NAME_TEMPLATE = "{stem}_chunk_{index}.md"
CHUNK_GLOB = "*_chunk_*.md"
DONE_SUFFIX = ".done"                             # a synthesised chunk is renamed to <name>.done
LOG_NAME_TEMPLATE = "log_{session_id}_{timestamp}.md"
SESSION_ID_PREFIX = "phd-"

# Query queue (one file per query) and the forager's keyword harvest
QUERY_FILE_SUFFIX = ".md"
PART_SUFFIX = ".part"                             # files are written as <name>.part, then renamed
HARVESTED_FILE = DRIVE_QUERIES / "harvested_keywords.md"

# Document types that will be taken from unprocessed/ directory and chunked
SUPPORTED_EXTENSIONS = ['.pdf', '.txt', '.md']

# Model files on Drive are named <model>.gguf (see the setup notebook)
MODEL_FILE_SUFFIX = ".gguf"

# ------------------------------------------------------------------------------
# 2c. Ollama and the models
# ------------------------------------------------------------------------------
OLLAMA_HOST = "http://127.0.0.1:11434"
OLLAMA_INSTALL_COMMAND = "curl -fsSL https://ollama.com/install.sh | sh"
OLLAMA_LOG_FILE = LOCAL_DIR / "ollama-server.log"
OLLAMA_START_TIMEOUT = 60                         # seconds to wait for the server to come up
OLLAMA_PING_TIMEOUT = 2                           # seconds
MODELFILE_DIR = LOCAL_DIR
MODELFILE_PREFIX = "Modelfile-"

AGENT_MODEL = "local-analyst"
EMBED_MODEL = "local-embed"
# Context is a measure of how much data the model can hold and use as a reference.  Maintaining the same context size throughout stops having to reload the model.
CONTEXT_SIZE = 8192
EMBED_CONTEXT_SIZE = 512
# Temperature is a measure of the model's 'creativity', where lower is more deterministic and higher is more creative.
TEMP_LATERAL = 0.9    # lateral keyword brainstorming (creative)
TEMP_QUERY = 0.5      # search-query generation (balanced)
TEMP_ANALYSIS = 0.2   # chunk synthesis, memory compression, focus evolution (precise)
EMBED_TEMPERATURE = 0.0

# Used when the memory / instructions files cannot be read
DEFAULT_RESEARCH_FOCUS = "computational archaeology and predictive modelling"
DEFAULT_INSTRUCTIONS = "Analyze the provided text."
DEFAULT_MEMORY_TEXT = "computational archaeology"

# Drive synchronisation
SYNC_INTERVAL = 3000                              # seconds between pushes of memories/logs to Drive
SYNC_RETRY_ATTEMPTS = 4                           # retries when Drive is locking a file
SYNC_RETRY_DELAY = 2                              # seconds between those retries

# Main loop
LOOP_IDLE_SLEEP = 60                              # seconds to wait when there is nothing to do
LOOP_ERROR_SLEEP = 5                              # seconds to wait after an error
FILES_PER_CHUNKING_BATCH = 50                     # documents chunked before switching back to synthesis
SYNTHESES_PER_COMPRESSION = 50                    # memories compressed (and the research focus refreshed) every N syntheses
NUM_META_SYNTHESES = 3                            # most recent compressed summaries used to refresh the research focus

# Query generator (background thread)
QUERY_QUEUE_TARGET = 5                            # top the queue up whenever it holds fewer queries than this
QUERY_CHECK_INTERVAL = 60                         # seconds between queue checks
QUERY_MAX_WORDS = 6                               # hard cap on words per generated query
RECENT_QUERIES_COUNT = 15                         # past queries shown to the model so it does not repeat them
HARVEST_RECENT_LINES = 5                          # forager keyword-harvest lines read per cycle

# Lateral-thinking excerpts
LATERAL_SNIPPET_CHARS = 1500                      # taken from the start of each processed paper
MIN_SNIPPET_CHARS = 100                           # shorter snippets are ignored
MIN_AGGREGATED_CHARS = 200                        # give up if the combined excerpts are shorter than this
MAX_LATERAL_CHARS = int(CONTEXT_SIZE * 2.2)       # ~55% of the context window (~4 chars/token); the rest is for the prompt and the answer

# Chunking
SEMANTIC_BREAKPOINT_PERCENTILE = 25               # split where sentence-to-sentence similarity drops below this percentile
MAX_CHUNK_CHARS = 4000
MIN_CHUNK_WORDS = 40
MIN_CHUNK_CHARS = 200
PDF_HEADER_MARGIN_PCT = 0.08                      # top of each page ignored
PDF_FOOTER_MARGIN_PCT = 0.10                      # bottom of each page ignored
REFERENCES_HEADING_MAX_CHARS = 50                 # a block shorter than this that matches the pattern below ends the paper
REFERENCES_HEADING_PATTERN = r'^((?:\d+|[IVXLCDM]+)\.?\s*)?(references|bibliography|works cited|literature cited)$'
# Chunks matching either pattern are metadata rather than content and are discarded
CHUNK_REJECT_PATTERN_1 = r"(Publication date|Reference format|Vol\.\s*\d+)"
CHUNK_REJECT_PATTERN_2 = r"([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+|University of)"

# The research-focus label written into MEMORY.md
FOCUS_LABEL = "**Research Focus:**"
# ==============================================================================
# 3. PROMPTS
#    Every prompt sent to the model. {placeholders} are filled in when the prompt is used.
# ==============================================================================
LATERAL_PROMPT = (
    "Read the following aggregated excerpts from papers previously integrated into our research:\n"
    "```text\n{aggregated_text}\n```\n\n"
    "Based on this text and our core focus ({research_focus}), brainstorm 3 to 5 highly unexpected, "
    "novel, and lateral academic keywords, algorithms, or methodologies we should search for to expand our research horizon.\n"
    "Output ONLY the keywords separated by spaces. No commas, no filler text."
    "CRITICAL DIRECTIVE: do not join words that should be seperate."
)

QUERY_PROMPT = (
    "You are an expert academic librarian. Generate a 3 to 5 term 'bag-of-words' search query.\n"
    "Combine 2 core keywords from the Research Focus with 1-2 bridging lateral terms.\n\n"
    "Heavily prioritize this theme: {nudge_text}\n\n"
    "Use these lateral concepts as inspiration: {lateral_concepts}\n\n"
    "Keywords from recent search results that indicate useful keywords: {harvested_keywords}\n\n"
    "CRITICAL RULES:\n"
    "1. Generate exactly 3 to 5 space-separated keywords/phrases ON A SINGLE LINE.\n"
    "2. DO NOT use Boolean operators (AND, OR, NOT), quotes, or parentheses.\n"
    "3. DO NOT repeat past queries: [{recent_queries}]\n\n"
    "Output ONLY the raw query string. No conversational text.\n\n"
    "Focus:\n{research_focus}"
)

SYNTHESIS_PROMPT = (
    "Act as a researcher, with this focus:\n{research_focus}\n\n"
    "Your instructions are:\n{instructions}\n\n"
    "**CRITICAL DIRECTIVES:**\n"
    "1. You must conclude your synthesis with the exact heading `### Auto-Memory:` followed by exactly one sentence stating the specific methodology, insight, or parameter to be added to our permanent literature review.\n"
    "2. Output ONLY the final response. Do not include any conversational filler or <think> tags.\n\n"
)

COMPRESSION_PROMPT = (
    "Act as an expert researcher. Synthesize the following {count} methodological "
    "insights into 3 to 5 thematic bullet points. Focus strictly on overarching themes.\n\n"
    "Output ONLY the bullet points. Do not include introductory text or <think> tags.\n\n"
    "{insights}"
)

EVOLUTION_PROMPT = (
    "Analyze the following condensed thematic insights extracted from recent literature:\n\n"
    "{all_insights}\n\n"
    "TASK:\n"
    "Synthesize these methodologies into exactly three concise bullet points representing the core academic themes of our evolving research domain. "
    "Start each bullet with a bolded high-level concept.\n"
    "Output ONLY the bullet points. Do not include introductory text or <think> tags.\n"
)

# ==============================================================================
# 3. CORE OLLAMA MANAGEMENT
# ==============================================================================
env = os.environ.copy()
env["OLLAMA_MODELS"] = str(LOCAL_OLLAMA)

local_ollama = Client(host=OLLAMA_HOST)

def sync_directories(source_dir: Path | str, dest_dir: Path | str, delete: bool = False, ignore: tuple = ()) -> None:
    """Safely synchronizes directories on Windows, bypassing Google Drive file locks."""
    src, dst = Path(source_dir), Path(dest_dir)
    if not src.exists(): return

    for attempt in range(SYNC_RETRY_ATTEMPTS):
        try:
            if delete and dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns(*ignore) if ignore else None)
            break  # Success
        except (PermissionError, OSError) as e:
            # If Google Drive is actively syncing/locking a file, wait and retry
            print(f"[Sync] File lock detected by Google Drive sync. Retrying in {SYNC_RETRY_DELAY}s...")
            time.sleep(SYNC_RETRY_DELAY)
        except Exception as e:
            print(f"[Sync] Non-fatal sync warning for {src}: {e}")
            break

if not shutil.which("ollama"):
    subprocess.run(OLLAMA_INSTALL_COMMAND, shell=True, check=True)

def ollama_ready():
    try: return requests.get(f"{OLLAMA_HOST}/api/tags", timeout=OLLAMA_PING_TIMEOUT).ok
    except Exception: return False

if not ollama_ready():
    ollama_log = open(OLLAMA_LOG_FILE, "w")
    subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, env=env)
    for _ in range(OLLAMA_START_TIMEOUT):
        if ollama_ready(): break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama server failed to start.")

def ingest_model(model_name: str, context_size: int, temperature: float):
    print(f"[System] Ingesting '{model_name}' (Context: {context_size})...")
    filename = f"{model_name}{MODEL_FILE_SUFFIX}"
    drive_gguf = DRIVE_MODELS / filename
    if not drive_gguf.exists():
        raise FileNotFoundError(f"[System] CRITICAL ERROR: Missing required GGUF on Drive: {drive_gguf}")
    modelfile_path = MODELFILE_DIR / f"{MODELFILE_PREFIX}{model_name}"
    modelfile_path.write_text(f"FROM {drive_gguf}\nPARAMETER num_ctx {context_size}\nPARAMETER temperature {temperature}\n", encoding="utf-8")
    subprocess.run(["ollama", "create", model_name, "-f", str(modelfile_path)], env=env, check=True)
    modelfile_path.unlink(missing_ok=True)
    print(f"[System] {model_name} successfully ingested and ready.")

def unload_all_models():
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/ps", timeout=OLLAMA_PING_TIMEOUT)
        if response.status_code == 200:
            active_models = response.json().get("models", [])
            if not active_models: return
            for model in active_models:
                subprocess.run(["ollama", "stop", model.get("name")], stdout=subprocess.DEVNULL)
    except Exception as e:
        print(f"[System] Error during VRAM purge: {e}")

def clean_llm_response(raw_text: str) -> str:
    clean_text = raw_text.strip()
    if '</think>' in clean_text: clean_text = clean_text.split('</think>')[-1]
    elif '</thought>' in clean_text: clean_text = clean_text.split('</thought>')[-1]
    clean_text = re.sub(r'(?is)(?:<think>|<thought>|Thinking Process:|Thought:|Reasoning:).*?(?:</think>|</thought>|$)', '', clean_text).strip()
    return clean_text

# ==============================================================================
# 4. BACKGROUND PROMPT GENERATION (THE "BRAIN")
# ==============================================================================
def query_key(query: str) -> str:
    return hashlib.md5(" ".join(query.lower().split()).encode("utf-8")).hexdigest()[:8]

def _all_query_files() -> list:
    files = []
    for d in (DRIVE_DONE, DRIVE_CLAIMED, DRIVE_PENDING):
        files.extend(d.glob(f"*{QUERY_FILE_SUFFIX}"))
    return sorted(files, key=lambda p: p.name)   # names start with a timestamp -> chronological

def recent_query_texts(n: int = RECENT_QUERIES_COUNT) -> list:
    texts = []
    for f in _all_query_files()[-n:]:
        try: t = f.read_text(encoding="utf-8").strip()
        except OSError: continue
        if t: texts.append(t)
    return texts

def enqueue_query(query: str) -> bool:
    """Atomically drops one query file into pending/. Returns False if it duplicates a known query."""
    key = query_key(query)
    if any(f.name.endswith(f"_{key}{QUERY_FILE_SUFFIX}") for f in _all_query_files()):
        return False
    name = f"{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}_{key}{QUERY_FILE_SUFFIX}"
    part = DRIVE_PENDING / (name + PART_SUFFIX)    # the forager only globs *.md, so it never sees .part files
    part.write_text(query.strip() + "\n", encoding="utf-8")
    os.replace(part, DRIVE_PENDING / name)
    return True

def pull_new_unprocessed() -> None:
    """Copies files the forager has dropped on Drive that we don't already hold. Never deletes or mirrors."""
    for f in DRIVE_UNPROCESSED.iterdir():
        if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS and not (LOCAL_UNPROCESSED / f.name).exists():
            try: shutil.copy2(f, LOCAL_UNPROCESSED / f.name)
            except (PermissionError, OSError) as e:
                print(f"[Sync] Could not pull {f.name} yet (Drive lock?): {e}")

def commit_processed(names: list) -> None:
    """Per file: archive to Drive processed/, and only then remove it from Drive unprocessed/."""
    for name in names:
        src = LOCAL_PROCESSED / name
        try:
            if src.exists(): shutil.copy2(src, DRIVE_PROCESSED / name)
            (DRIVE_UNPROCESSED / name).unlink(missing_ok=True)
        except (PermissionError, OSError) as e:
            print(f"[Sync] Could not commit {name}; will be re-processed next time: {e}")

def autonomous_internal_exploration(current_focus: str) -> str:
    """Aggregates multiple processed abstracts to generate lateral search concepts with high temperature."""
    processed_files = list(LOCAL_PROCESSED.glob("*.md")) + list(LOCAL_PROCESSED.glob("*.pdf"))
    if not processed_files:
        return ""

    print(f"[Lateral thought] Aggregating past papers for lateral thinking...")

    # Shuffle to ensure a diverse, unpredictable sample of papers is read every loop
    random.shuffle(processed_files)
    aggregated_text = ""

    try:
        for file_path in processed_files:
            if len(aggregated_text) >= MAX_LATERAL_CHARS:
                break

            snippet = ""
            try:
                if file_path.suffix.lower() == '.pdf':
                    doc = fitz.open(file_path)
                    # Grab the first chunk of text (usually covers the abstract/intro)
                    snippet = doc[0].get_text("text")[:LATERAL_SNIPPET_CHARS] if len(doc) > 0 else ""
                    doc.close()
                else:
                    snippet = file_path.read_text(encoding="utf-8")[:LATERAL_SNIPPET_CHARS]
            except Exception:
                continue # Skip unreadable files gracefully

            if len(snippet) > MIN_SNIPPET_CHARS:
                aggregated_text += f"\n--- Excerpt from {file_path.name} ---\n{snippet.strip()}\n"

        if len(aggregated_text) < MIN_AGGREGATED_CHARS:
            return ""

        lateral_prompt = LATERAL_PROMPT.format(aggregated_text=aggregated_text[:MAX_LATERAL_CHARS], research_focus=current_focus)

        # High temperature (0.9) forces maximum creative drift and lateral connections
        response = local_ollama.generate(
            model=AGENT_MODEL,
            prompt=lateral_prompt,
            options={"num_ctx": CONTEXT_SIZE, "temperature": TEMP_LATERAL}
        )

        new_concepts = clean_llm_response(response.get("response", "")).strip('"\'')
        print(f"[Lateral thought] New concepts found: {new_concepts}")
        return new_concepts

    except Exception as e:
        print(f"[Lateral thought] Internal exploration failed: {e}")
        return ""

def query_generator_loop():
    """Runs continuously in the background to replenish the query queue for the local laptop."""

    print("[System] Autonomous Query Generator thread active in background...")

    harvest_file = HARVESTED_FILE
    memory_file = DRIVE_MEMORY_FILE
    nudge_file = DRIVE_NUDGE_FILE

    while True:
        try:
            pending = list(DRIVE_PENDING.glob(f"*{QUERY_FILE_SUFFIX}"))

            if len(pending) < QUERY_QUEUE_TARGET:
                # 1. Gather Context
                memory_text = memory_file.read_text(encoding="utf-8") if memory_file.exists() else DEFAULT_MEMORY_TEXT
                domain_match = re.search(r'## Academic Domain(.*?)(?:##|$)', memory_text, re.DOTALL)
                research_focus = domain_match.group(1).strip() if domain_match else memory_text[:500]
                nudge_text = nudge_file.read_text(encoding="utf-8").strip() if nudge_file.exists() else ""

                recent_harvested = ""
                if harvest_file.exists():
                    lines = [line for line in harvest_file.read_text(encoding="utf-8").splitlines() if line.startswith("- ")]
                    recent_harvested = ", ".join(lines[-HARVEST_RECENT_LINES:])

                past_queries = recent_query_texts()
                recent_queries = ", ".join(past_queries) if past_queries else "None"

                # 2. Go brainstorming
                lateral_concepts = autonomous_internal_exploration(research_focus)

                # 3. Construct the Prompt
                query_prompt = QUERY_PROMPT.format(nudge_text=nudge_text, lateral_concepts=lateral_concepts, harvested_keywords=recent_harvested or "None", recent_queries=recent_queries, research_focus=research_focus)

                # 4. Ask LLM and save with double newlines for readability
                response = local_ollama.generate(
                    model=AGENT_MODEL,
                    prompt=query_prompt,
                    options={"num_ctx": CONTEXT_SIZE, "temperature": TEMP_QUERY}
                    )

                clean_query = clean_llm_response(response.get("response", "")).strip('"\'')
                clean_query = " ".join(clean_query.replace('\n', ' ').split()[:QUERY_MAX_WORDS])

                if clean_query:
                    if enqueue_query(clean_query):
                        print(f"[Query generation] Dropped new query to Drive Queue: '{clean_query}'")
                    else:
                        print(f"[Query generation] Skipped duplicate of a recent query: '{clean_query}'")

        except Exception:
            pass # Fail silently and try again on next loop

        time.sleep(QUERY_CHECK_INTERVAL) # Wait before checking the queue again

# Start the query generator daemon
threading.Thread(target=query_generator_loop, daemon=True).start()
# ==============================================================================
# 5. SYNTHESIS WORKER FUNCTIONS
# ==============================================================================
def apply_research_focus(memory_text: str, new_focus_text: str):
    """
    Creates or updates ONLY the Research Focus in MEMORY.md; every other line is returned unchanged.
      - a Research Focus already exists -> its block (label + the lines directly under it) is replaced in place
      - none exists                     -> one is created at the end of '## Academic Domain'
                                           (or the section itself is created if that is missing too)
    Returns (updated_text, "updated" | "created").
    """
    block = [FOCUS_LABEL] + [l.strip() for l in new_focus_text.strip().splitlines() if l.strip()]
    lines = memory_text.split("\n")
    is_heading = lambda l: re.match(r"^[ \t]*##(?!#)", l) is not None
    label = re.compile(r"^[ \t]*(?:[*\-][ \t]+)?\*\*Research Focus:?\*\*:?", re.IGNORECASE)

    # 1. Update in place. The block ends at the first blank line or the next heading.
    for i, line in enumerate(lines):
        if label.match(line):
            j = i + 1
            while j < len(lines) and lines[j].strip() and not is_heading(lines[j]): j += 1
            return "\n".join(lines[:i] + block + lines[j:]), "updated"

    # 2. Create at the end of the Academic Domain section (after its last non-blank line).
    start = next((i for i, l in enumerate(lines) if re.match(r"^[ \t]*##[ \t]+Academic Domain\b", l, re.IGNORECASE)), None)
    if start is not None:
        end = next((k for k in range(start + 1, len(lines)) if is_heading(lines[k])), len(lines))
        k = end
        while k - 1 > start and not lines[k - 1].strip(): k -= 1
        return "\n".join(lines[:k] + [""] + block + lines[k:]), "created"

    # 3. No Academic Domain section at all: create it, ahead of Synthesis Rules if that exists.
    section = ["## Academic Domain"] + block + [""]
    idx = next((i for i, l in enumerate(lines) if re.match(r"^[ \t]*##[ \t]+Synthesis Rules", l, re.IGNORECASE)), None)
    if idx is not None:
        return "\n".join(lines[:idx] + section + lines[idx:]), "created"
    base = memory_text.rstrip("\n")
    return (base + "\n\n" if base else "") + "\n".join(section) + "\n", "created"

def evolve_academic_domain(max_syntheses: int = 3):
    memory_file = DRIVE_MEMORY_FILE
    history_file = FOCUS_HISTORY_FILE
    meta_files = sorted(list(LOCAL_MEMORY.glob(META_SYNTHESIS_GLOB)), reverse=True)
    if not meta_files: return

    current_memory = memory_file.read_text(encoding="utf-8") if memory_file.exists() else ""
    recent_files = meta_files[:max_syntheses]
    all_insights = "\n\n".join([f.read_text(encoding="utf-8") for f in recent_files])

    evolution_prompt = EVOLUTION_PROMPT.format(all_insights=all_insights)

    try:
        response = local_ollama.generate(
            model=AGENT_MODEL,
            prompt=evolution_prompt,
            options={"num_ctx": CONTEXT_SIZE, "temperature": TEMP_ANALYSIS}
            )

        raw_response = response.get("response", "")
        cleaned_text = clean_llm_response(raw_response)
        new_focus_text = '\n'.join(line.strip() for line in cleaned_text.splitlines() if line.strip())

        if not new_focus_text: return
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

        history_block = f"### Evolution [{timestamp}]\n{new_focus_text}\n\n"
        if history_file.exists():
            with open(history_file, "a", encoding="utf-8") as f: f.write(history_block)
        else:
            history_file.write_text(f"# Evolved Research Focus History\n\n{history_block}", encoding="utf-8")

        # Broader Interests (and everything else) is never rewritten; only the Research Focus is created/updated.
        updated_memory, action = apply_research_focus(current_memory, new_focus_text)
        broader = re.search(r"^.*\*\*Broader Interests:?\*\*.*$", current_memory, re.IGNORECASE | re.MULTILINE)
        if broader and broader.group(0) not in updated_memory:
            print("[Research focus] Aborted: the update would have altered Broader Interests. MEMORY.md left as it was.")
            return

        memory_file.write_text(updated_memory, encoding="utf-8")
        LOCAL_MEMORY_FILE.write_text(updated_memory, encoding="utf-8")
        print(f"[Research focus] Research Focus {action} in MEMORY.md" + ("; Broader Interests left untouched." if broader else "."))
    except Exception as e:
        print(f"[Research focus] Reflection failed:\n{e}")

def is_valid_chunk(chunk_text: str) -> bool:
    if re.search(CHUNK_REJECT_PATTERN_1, chunk_text, re.IGNORECASE): return False
    if re.search(CHUNK_REJECT_PATTERN_2, chunk_text, re.IGNORECASE): return False
    if len(chunk_text.split()) < MIN_CHUNK_WORDS: return False
    if len(chunk_text.strip()) < MIN_CHUNK_CHARS: return False
    return True

def semantic_chunker(text: str, embed_model: str = EMBED_MODEL, threshold_percentile: int = SEMANTIC_BREAKPOINT_PERCENTILE, max_chunk_chars: int = MAX_CHUNK_CHARS) -> list:
    sentences = sent_tokenize(text)
    if len(sentences) <= 2: return [text]
    try:
        response = local_ollama.embed(model=embed_model, input=sentences)
        embeddings = response.get("embeddings", [])
    except Exception as e:
        return []

    if not embeddings or len(embeddings) != len(sentences): return []

    similarities = [cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0] for i in range(len(embeddings) - 1)]
    breakpoint_threshold = np.percentile(similarities, threshold_percentile)

    chunks = []
    current_chunk = [sentences[0]]
    current_length = len(sentences[0])

    for i, sim in enumerate(similarities):
        next_sentence = sentences[i+1]
        next_length = len(next_sentence)

        if sim < breakpoint_threshold or (current_length + next_length) > max_chunk_chars:
            chunks.append(" ".join(current_chunk))
            current_chunk = [next_sentence]
            current_length = next_length
        else:
            current_chunk.append(next_sentence)
            current_length += next_length + 1

    if current_chunk: chunks.append(" ".join(current_chunk))
    return chunks

def process_file(target_file: Path, header_margin_pct: float = PDF_HEADER_MARGIN_PCT, footer_margin_pct: float = PDF_FOOTER_MARGIN_PCT):
    print(f"[Chunking] {target_file.name}")
    try:
        if target_file.suffix.lower() == ".pdf":
            doc = fitz.open(target_file)
            clean_text_blocks = []
            stop_pattern = re.compile(REFERENCES_HEADING_PATTERN, re.IGNORECASE)
            parsing_active = True

            for page in doc:
                if not parsing_active: break
                page_height = page.rect.height
                top_limit = page_height * header_margin_pct
                bottom_limit = page_height * (1 - footer_margin_pct)
                blocks = page.get_text("blocks")

                for b in blocks:
                    if b[6] == 0:
                        text_content = b[4].strip()
                        if len(text_content) < REFERENCES_HEADING_MAX_CHARS and stop_pattern.match(text_content):
                            parsing_active = False
                            break
                        y0_coordinate, y1_coordinate = b[1], b[3]
                        if y0_coordinate > top_limit and y1_coordinate < bottom_limit:
                            if not text_content.isdigit(): clean_text_blocks.append(text_content)

            raw_text = "\n\n".join(clean_text_blocks)
        else:
            raw_text = target_file.read_text(encoding="utf-8")

    except Exception as e:
        print(f"[Chunking] Read failed: {e}")
        return False

    if raw_text.strip():
        raw_chunks = semantic_chunker(raw_text)
        valid_chunks = [chunk for chunk in raw_chunks if is_valid_chunk(chunk)]

        for i, chunk in enumerate(valid_chunks, 1):
            (LOCAL_TEMP / CHUNK_NAME_TEMPLATE.format(stem=target_file.stem, index=i)).write_text(chunk, encoding="utf-8")

    shutil.move(str(target_file), str(LOCAL_PROCESSED / target_file.name))
    return True

def cache_prompt_context_to_ram() -> str:
    memory_path = LOCAL_MEMORY_FILE
    instructions_path = LOCAL_INSTRUCTIONS_FILE
    print("[System] Caching system prompts to RAM...")
    research_focus = memory_path.read_text(encoding="utf-8").strip() if memory_path.exists() else DEFAULT_RESEARCH_FOCUS
    instructions = instructions_path.read_text(encoding="utf-8").strip() if instructions_path.exists() else DEFAULT_INSTRUCTIONS

    synthesis_prompt = SYNTHESIS_PROMPT.format(research_focus=research_focus, instructions=instructions)
    return synthesis_prompt

def ask_agent(chunk_text: str, cached_system_prompt: str, session_id: str = None):
    combined_prompt = f"{cached_system_prompt}## Target Semantic Chunk\n{chunk_text}"
    if not session_id: session_id = SESSION_ID_PREFIX + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

    try:
        response = local_ollama.generate(
            model=AGENT_MODEL,
            prompt=combined_prompt,
            options={"num_ctx": CONTEXT_SIZE, "temperature": TEMP_ANALYSIS}
            )

        clean_response = clean_llm_response(response.get("response", "").strip())
        memory_match = re.search(r'Auto-Memory:\s*(.*)', clean_response, re.IGNORECASE | re.DOTALL)

        if memory_match:
            memory_sentence = memory_match.group(1).strip().split('\n')[0].strip()
            print(f"[Summary] {memory_sentence[:80]}...")
            timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
            content = (
                f"# Session: {session_id}\n**Timestamp:** {timestamp}\n\n"
                f"## Target Chunk Processed\n```text\n{chunk_text}\n```\n\n"
                f"## Agent Synthesis\n{clean_response}\n\n"
                f"## Extracted Auto-Memory\n{memory_sentence}\n"
            )
            local_file = LOCAL_LOGS / LOG_NAME_TEMPLATE
            local_file.write_text(content, encoding="utf-8")
            with open(COMPILED_MEMORIES_FILE, "a", encoding="utf-8") as f:
                f.write(f"- {memory_sentence}\n")
        else:
            pass
    except Exception as e:
        print(f"[Summary] Error during Ollama query:\n{e}")
    return session_id

def compress_memories():
    master_file = COMPILED_MEMORIES_FILE
    if not master_file.exists(): return
    lines = [line for line in master_file.read_text(encoding="utf-8").split('\n') if line.strip()]
    if len(lines) < SYNTHESES_PER_COMPRESSION: return

    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    meta_file = LOCAL_MEMORY / META_SYNTHESIS_TEMPLATE.format(timestamp=timestamp)
    target_lines = lines[:SYNTHESES_PER_COMPRESSION]
    remaining_lines = lines[SYNTHESES_PER_COMPRESSION:]

    compression_prompt = COMPRESSION_PROMPT.format(count=SYNTHESES_PER_COMPRESSION, insights="\n".join(target_lines))

    try:
        response = local_ollama.generate(
            model=AGENT_MODEL,
            prompt=compression_prompt,
            options={"num_ctx": CONTEXT_SIZE, "temperature": TEMP_ANALYSIS}
            )

        clean_response = clean_llm_response(response.get("response", "").strip())
        if clean_response:
            formatted_bullets = '\n'.join(line.strip() for line in clean_response.split('\n') if line.strip())
            meta_file.write_text(f"### Synthesis [{timestamp}]\n{formatted_bullets}\n\n", encoding="utf-8")
            master_file.write_text("\n".join(remaining_lines) + "\n", encoding="utf-8")
            print(f"[Compress memories] Compressed {SYNTHESES_PER_COMPRESSION} insights into {meta_file.name}")
    except Exception as e:
        print(f"[Compress memories] Compression failed: {e}")
# ==============================================================================
# 7. MASTER SYNTHESIS LOOP
# ==============================================================================
def run_synthesis_loop(evolve: bool = True, num_meta_syntheses: int = NUM_META_SYNTHESES):
    print("[System] Starting continuous evolving research agent...")

    sync_directories(DRIVE_WORKSPACE, LOCAL_WORKSPACE)
    sync_directories(DRIVE_PERSONALITY, LOCAL_PERSONALITY, ignore=("queries",))   # queues live on Drive only

    print("[System] Registering required models for this session...")
    ingest_model(AGENT_MODEL, context_size=CONTEXT_SIZE, temperature=TEMP_ANALYSIS)
    ingest_model(EMBED_MODEL, context_size=EMBED_CONTEXT_SIZE, temperature=EMBED_TEMPERATURE)

    current_phase = None
    processed_drive_chunks = []
    pending_local_deletions = []
    failed_files = set()   # unreadable files: skip for this session, leave on Drive

    if evolve:
        current_phase = "EVOLUTION"
        print("\nINITIATING EVOLUTION PHASE")
        unload_all_models()
        evolve_academic_domain(max_syntheses=num_meta_syntheses)

    cached_system_prompt = cache_prompt_context_to_ram()
    last_drive_sync = time.time()
    synthesis_counter = 0

    while True:
        try:
            chunk_files = list(LOCAL_TEMP.glob(CHUNK_GLOB))

            if not chunk_files:
                if current_phase == "SYNTHESIS" and processed_drive_chunks:
                    print("[System] Local chunks exhausted. Pushing to Google Drive...")
                    sync_directories(LOCAL_MEMORY, DRIVE_MEMORY)
                    sync_directories(LOCAL_LOGS, DRIVE_LOGS)

                    for chunk_name in processed_drive_chunks:
                        drive_chunk = DRIVE_TEMP / chunk_name
                        if drive_chunk.exists(): drive_chunk.unlink()
                    processed_drive_chunks.clear()
                    last_drive_sync = time.time()

                pull_new_unprocessed()
                raw_files = [f for f in LOCAL_UNPROCESSED.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS and f.name not in failed_files]

                if not raw_files:
                    time.sleep(LOOP_IDLE_SLEEP)
                    continue

                if current_phase != "CHUNKING":
                    print("\nINITIATING BATCH CHUNKING PHASE")
                    unload_all_models()
                    current_phase = "CHUNKING"

                batch_done = []
                for target_file in raw_files[:FILES_PER_CHUNKING_BATCH]:
                    if process_file(target_file): batch_done.append(target_file.name)
                    else: failed_files.add(target_file.name)

                sync_directories(LOCAL_TEMP, DRIVE_TEMP)
                commit_processed(batch_done)   # per file: archive, then remove from Drive unprocessed/

                chunk_files = list(LOCAL_TEMP.glob(CHUNK_GLOB))
                if not chunk_files: continue

            else:
                random.shuffle(chunk_files)
                next_chunk_path = chunk_files[0]
                chunk_text = next_chunk_path.read_text(encoding="utf-8")

                if current_phase != "SYNTHESIS":
                    print("\nINITIATING SYNTHESIS PHASE")
                    unload_all_models()
                    current_phase = "SYNTHESIS"

                session_id = ask_agent(chunk_text=chunk_text, cached_system_prompt=cached_system_prompt)

                if session_id:
                    synthesis_counter += 1
                    processed_drive_chunks.append(next_chunk_path.name)

                    done_path = next_chunk_path.with_suffix(DONE_SUFFIX)
                    next_chunk_path.rename(done_path)
                    pending_local_deletions.append(done_path)

                    if synthesis_counter % SYNTHESES_PER_COMPRESSION == 0:
                        compress_memories()
                        evolve_academic_domain(max_syntheses=num_meta_syntheses)
                        cached_system_prompt = cache_prompt_context_to_ram()
                        for chunk_to_delete in pending_local_deletions: chunk_to_delete.unlink(missing_ok=True)
                        pending_local_deletions.clear()

                    current_time = time.time()
                    if current_time - last_drive_sync >= SYNC_INTERVAL:
                        print("[System] Synchronization threshold reached. Pushing to Drive...")
                        sync_directories(LOCAL_MEMORY, DRIVE_MEMORY)
                        sync_directories(LOCAL_LOGS, DRIVE_LOGS)

                        for chunk_name in processed_drive_chunks:
                            drive_chunk = DRIVE_TEMP / chunk_name
                            if drive_chunk.exists(): drive_chunk.unlink()
                        processed_drive_chunks.clear()
                        last_drive_sync = current_time
                else:
                    time.sleep(LOOP_ERROR_SLEEP)

        except Exception as e:
            time.sleep(LOOP_ERROR_SLEEP)

if __name__ == "__main__":
    run_synthesis_loop()